# Tutorial: Calibrating Systematic Poking Material Parameters from Reaction Forces (PyTorch)

**Goal**: fit Systematic Poking material parameters from a set of static
reaction-force curves so that the fitted material's reactions match a
Neo-Hookean target -- this time with a differentiable equilibrium layer.

This notebook implements the reaction-force calibration entirely with
PyTorch: the physics, mesh, load cases, and Gauss-Newton fit follow the
experiment exactly, but the equilibrium solve is wrapped in
`pypgo.fem.torch.StaticEquilibriumLayer`. The layer's backward pass
implements the implicit-function-theorem adjoint internally, so the
Jacobian of every reaction with respect to the material parameters comes
from autograd instead of a hand-written formula.

## Outline

1. Build a 2×2×2 hexahedral grid;
2. Construct the elastic energy operator (material definition / binding);
3. Define load cases and wrap each one in a differentiable equilibrium layer;
4. Generate "ground truth" reactions with the target material;
5. Residuals and the reaction Jacobian via autograd;
6. Gauss-Newton fit in log space;
7. Results and visualization.

## 0. Setup

The cell below adds the repository root to `sys.path` so `pypgo` is importable
from any directory, and sets the C++ backend's log level to `off` so spdlog
does not flood the output. PyTorch is an optional dependency
(`pip install pypgo[torch]`); the torch layer lives in `pypgo.fem.torch` and is
imported lazily.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pypgo").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch

import pypgo as pgo
import pypgo.fem as pf
import pypgo.solver as ps
from pypgo.fem.torch import StaticEquilibriumLayer

# The C++ backend logs through spdlog to stdout; silence it for this notebook.
# Restore at any time with pgo.set_log_level("info").
pgo.set_log_level("off")

## 1. Experiment configuration and mesh

All experimental constants live at the top. The mesh is a unit cube split
into `GRID_SIZE^3` hexahedra (8 corners each) -- tiny, but enough to produce
meaningful reaction-force curves.

In [ ]:
TARGET_E, TARGET_NU = 2.0e5, 0.35   # target Neo-Hookean material
INIT_E, INIT_NU = 1.2e5, 0.25       # starting guess for the fit
FORCE_SCALE = TARGET_E              # forces/parameters normalized by E
STATIC_TOL = 1.0e-6                 # equilibrium residual threshold
SMOOTHNESS = 1.0e-3                 # second-difference regularization weight
KNOT_COUNT = 17                     # stretch/volume knots (must be odd)
GRID_SIZE = 2                       # grid: divisions along each axis
SHEARS = (-0.8, -0.4, 0.4, 0.8)     # training shear magnitudes


def make_grid(n: int = GRID_SIZE):
    ids = np.arange((n + 1) ** 3).reshape((n + 1, n + 1, n + 1))
    vertices = np.array(
        [[i / n, j / n, k / n]
         for i in range(n + 1) for j in range(n + 1) for k in range(n + 1)],
        dtype=np.float64,
    )
    elements = np.array(
        [[ids[i, j, k], ids[i + 1, j, k], ids[i + 1, j + 1, k],
          ids[i, j + 1, k], ids[i, j, k + 1], ids[i + 1, j, k + 1],
          ids[i + 1, j + 1, k + 1], ids[i, j + 1, k + 1]]
         for i in range(n) for j in range(n) for k in range(n)],
        dtype=np.int64,
    )
    return vertices, elements


vertices, elements = make_grid()
print(f"vertices = {len(vertices)}   elements = {len(elements)}   "
      f"displacement dofs = {vertices.size}")

## 2. Building the elastic energy operator

`pypgo`'s material system has two layers:

- **Fixed channels** (`MaterialBinding`): per-element constants such as E, nu;
- **Optimizable channels** (`MaterialState`): the values being fitted, e.g.
  Systematic Poking's f''(knot) values and lambda.

The `DeformationEnergyOperator` is immutable and owns the binding, so we build
it **once** per material and share it across all load cases. Each forward pass
of the torch layer binds a fresh `MaterialState` to the operator; no per-case
energy construction is needed. The target material (Neo-Hookean) has zero
optimizable channels; the fitted material (Systematic Poking) has `f''(knot)`
values plus lambda.

In [ ]:
def fixed_channel_block(elastic, num_elements):
    """Element-major fixed channels for the MaterialBinding."""
    if elastic.name == "neo_hookean":
        return np.tile([TARGET_E, TARGET_NU], (num_elements, 1))
    if elastic.name == "systematic_poking":
        # Knot locations live in the definition; no per-element fixed channels
        return np.empty((num_elements, 0), dtype=np.float64)
    raise ValueError(f"unsupported fixed channels for {elastic.name}")


def make_operator(elastic):
    """Build a shared CubicLinear operator for one material definition."""
    volume = pgo.mesh.volume.VolumeMesh(
        pgo.mesh.CubicMeshData(*make_grid()),
        pgo.mesh.volume.ENuMaterial(E=TARGET_E, nu=TARGET_NU),
    )
    mesh = pf.SimulationMesh(volume)
    plastic = pf.VolumetricPlasticityDefinition(dofs=0)
    binding = pf.MaterialBinding(
        pf.ElasticMaterialBinding(
            elastic, mesh.num_elements,
            fixed_channel_block(elastic, mesh.num_elements)),
        pf.PlasticMaterialBinding(
            plastic, mesh.num_elements,
            np.empty((mesh.num_elements, 0), dtype=np.float64)),
    )
    return pf.DeformationEnergyOperator(
        mesh, binding,
        formulation=pf.CubicLinear(),
        options=pf.DeformationOptions(project_hessian_psd=False),
    )


knots = np.exp(np.linspace(np.log(0.5), np.log(2.0), KNOT_COUNT))
vol_knots = np.exp(np.linspace(-1.0, 1.0, KNOT_COUNT))
initial_params = np.concatenate([
    (INIT_E / (2 * (1 + INIT_NU))) * (1 + 1 / knots ** 2),
    [INIT_E * INIT_NU / ((1 + INIT_NU) * (1 - 2 * INIT_NU))],
])
systematic = pf.SystematicPokingDefinition(
    knots, KNOT_COUNT // 2, vol_knots, KNOT_COUNT // 2)

operator = make_operator(systematic)
target_operator = make_operator(pf.NeoHookeanDefinition())
print(f"systematic: {operator.num_dofs} dofs, "
      f"{operator.num_elastic_params} optimizable elastic channels/element")
print(f"neo-hookean: {target_operator.num_dofs} dofs, "
      f"{target_operator.num_elastic_params} optimizable elastic channels/element")

## 3. Load cases and reactions

Each load case is a set of **displacement boundary conditions** plus a
**reaction readout**:

- Uniaxial: bottom y fixed, top y compressed to `stretch`; `confined` also
  locks the sides, `free` adds only three minimal rigid pins;
- Shear: bottom fully fixed, top x translated by `shear`.

The reaction is `reaction_selector @ gradient`, i.e. the total force on the
top face: once the equilibrium is solved, the gradient *is* the reaction
force.

In [ ]:
from dataclasses import dataclass


@dataclass
class LoadCase:
    label: str
    protocol: str
    coordinate: float
    fixed_dofs: np.ndarray
    fixed_values: np.ndarray
    free_dofs: np.ndarray
    reaction_selector: np.ndarray
    initial_displacement: np.ndarray
    target: float = 0.0


def constraints_from_mapping(mapping, num_dofs):
    fixed_dofs = np.array(sorted(mapping), dtype=np.int64)
    fixed_values = np.array([mapping[int(d)] for d in fixed_dofs], dtype=np.float64)
    free_dofs = np.setdiff1d(np.arange(num_dofs, dtype=np.int64), fixed_dofs)
    return fixed_dofs, fixed_values, free_dofs


def free_uniaxial_pins(vertices, bottom):
    positions = vertices[bottom]
    centroid = positions.mean(axis=0)
    d2 = ((positions[:, 0] - centroid[0]) ** 2 +
          (positions[:, 2] - centroid[2]) ** 2)
    anchor = int(bottom[np.argmin(d2)])
    same_z = bottom[np.isclose(vertices[bottom, 2], vertices[anchor, 2])]
    candidates = same_z[same_z != anchor]
    if candidates.size == 0:
        candidates = bottom[bottom != anchor]
    rotation_pin = int(candidates[np.argmax(
        np.abs(vertices[candidates, 0] - vertices[anchor, 0]))])
    return anchor, rotation_pin


def _boundary_vertices(vertices):
    bottom = np.flatnonzero(np.isclose(vertices[:, 1], 0.0))
    top = np.flatnonzero(np.isclose(vertices[:, 1], 1.0))
    return bottom, top


def free_uniaxial_cases(vertices, stretches):
    """Bottom fixed in y; top compressed along y; three rigid pins on the bottom."""
    num_dofs = vertices.size
    bottom, top = _boundary_vertices(vertices)
    cases = []
    for stretch in stretches:
        mapping = {}
        for v in bottom:
            mapping[3 * int(v) + 1] = 0.0
        for v in top:
            mapping[3 * int(v) + 1] = stretch - 1.0
        anchor, rotation_pin = free_uniaxial_pins(vertices, bottom)
        mapping[3 * anchor] = 0.0
        mapping[3 * anchor + 2] = 0.0
        mapping[3 * rotation_pin + 2] = 0.0
        fixed_dofs, fixed_values, free_dofs = constraints_from_mapping(
            mapping, num_dofs)
        selector = np.zeros(num_dofs)
        selector[3 * top + 1] = 1.0
        initial = np.zeros(num_dofs)
        initial[1::3] = (stretch - 1.0) * vertices[:, 1]
        cases.append(LoadCase(
            label=f"free_uniaxial_{stretch:.3f}",
            protocol="free_uniaxial",
            coordinate=stretch,
            fixed_dofs=fixed_dofs, fixed_values=fixed_values,
            free_dofs=free_dofs, reaction_selector=selector,
            initial_displacement=initial))
    return cases


def confined_uniaxial_cases(vertices, stretches):
    """Bottom fixed in y; top compressed along y; all x/z DOFs pinned."""
    num_dofs = vertices.size
    bottom, top = _boundary_vertices(vertices)
    cases = []
    for stretch in stretches:
        mapping = {}
        for v in bottom:
            mapping[3 * int(v) + 1] = 0.0
        for v in top:
            mapping[3 * int(v) + 1] = stretch - 1.0
        for v in range(len(vertices)):
            mapping[3 * v] = 0.0
            mapping[3 * v + 2] = 0.0
        fixed_dofs, fixed_values, free_dofs = constraints_from_mapping(
            mapping, num_dofs)
        selector = np.zeros(num_dofs)
        selector[3 * top + 1] = 1.0
        initial = np.zeros(num_dofs)
        initial[1::3] = (stretch - 1.0) * vertices[:, 1]
        cases.append(LoadCase(
            label=f"confined_uniaxial_{stretch:.3f}",
            protocol="confined_uniaxial",
            coordinate=stretch,
            fixed_dofs=fixed_dofs, fixed_values=fixed_values,
            free_dofs=free_dofs, reaction_selector=selector,
            initial_displacement=initial))
    return cases


def simple_shear_cases(vertices, shears):
    """Bottom fully fixed; top translated along x by the shear amount."""
    num_dofs = vertices.size
    bottom, top = _boundary_vertices(vertices)
    cases = []
    for shear in shears:
        mapping = {}
        for v in bottom:
            for c in range(3):
                mapping[3 * int(v) + c] = 0.0
        for v in top:
            mapping[3 * int(v)] = shear
            mapping[3 * int(v) + 1] = 0.0
        fixed_dofs, fixed_values, free_dofs = constraints_from_mapping(
            mapping, num_dofs)
        selector = np.zeros(num_dofs)
        selector[3 * top] = 1.0
        initial = np.zeros(num_dofs)
        initial[0::3] = shear * vertices[:, 1]
        cases.append(LoadCase(
            label=f"simple_shear_{shear:+.3f}", protocol="simple_shear",
            coordinate=shear,
            fixed_dofs=fixed_dofs, fixed_values=fixed_values,
            free_dofs=free_dofs, reaction_selector=selector,
            initial_displacement=initial))
    return cases


def make_cases(vertices, knots):
    """Build the three protocol families explicitly and concatenate them."""
    stretches = [k for i, k in enumerate(knots) if i != len(knots) // 2]
    return (
        free_uniaxial_cases(vertices, stretches)
        + confined_uniaxial_cases(vertices, stretches)
        + simple_shear_cases(vertices, SHEARS)
    )

### Wrapping each load case in a differentiable equilibrium layer

`StaticEquilibriumLayer` is the torch bridge. It owns one load case's
immutable physics (shared operator, boundary conditions, inner Newton
optimizer, sparse-solver backend) and exposes the equilibrium solve as a
differentiable function:

- `forward(elastic_values, plastic_values)` solves the static equilibrium with
  the C++ Newton optimizer and returns the reaction
  `c^T grad E(u*)` (or `u*` itself if no selector is given);
- `backward` implements the implicit-function-theorem adjoint with the Hessian
  at the converged state and the operator's `material_vjp`, so Newton
  iterations are never unrolled into the autograd graph.

Inputs are element-major float64 CPU tensors of shape
`(num_elements, num_channels)`. Global parameters are broadcast by the caller
with `params.expand(num_elements, -1)`. The plastic argument is required and
uses shape `(num_elements, 0)` when the model has no plastic channels.

In [ ]:
OPTIMIZER = ps.NewtonOptimizer(
    max_iterations=200,
    termination=ps.AbsoluteTermination(abs_tolerance=1.0e-10),
)

cases = make_cases(operator.vertex_rest_positions, knots)
layers = [
    StaticEquilibriumLayer(
        energy_operator=operator,
        fixed_dofs=case.fixed_dofs,
        fixed_values=case.fixed_values,
        reaction_selectors=case.reaction_selector,
        inner_optimizer=OPTIMIZER,
        sparse_backend=ps.EigenLDLT(),
        initial_displacement=case.initial_displacement,
        residual_tol=STATIC_TOL,
    )
    for case in cases
]

num_elements = operator.num_elements
plastic_empty = torch.empty((num_elements, 0), dtype=torch.float64)

# One probe reaction through the first layer.
probe_params = torch.as_tensor(
    np.broadcast_to(
        initial_params, (num_elements, initial_params.size)).copy(),
    dtype=torch.float64)
probe = layers[0](probe_params, plastic_empty)
print(f"case: {cases[0].label}")
print(f"reaction / E = {probe.item() / FORCE_SCALE:.6f}")

### Visualizing the solved configurations

Before fitting, look at one solved configuration per protocol.  The
layer solves the equilibrium with the initial material and keeps the
converged displacement (`layer._last_displacement`); we rebuild the
deformed surface mesh directly and render it with `pgo.visualize` --
no intermediate OBJ files.  PyVista is an optional dependency: the
cell falls back to a message when it is not installed.

In [ ]:
def deformed_surface(energy_operator, displacement):
    """Surface TriMeshData of a deformed configuration."""
    deformed = (
        energy_operator.vertex_rest_positions
        + displacement.reshape(-1, 3))
    volume = pgo.mesh.volume.VolumeMesh(
        pgo.mesh.CubicMeshData(deformed, elements),
        pgo.mesh.volume.ENuMaterial(),
    )
    return volume.extract_surface_mesh()


try:
    pgo.visualize  # lazy import; requires pyvista
    HAS_VIZ = True
except Exception:
    HAS_VIZ = False

if HAS_VIZ:
    rest_volume = pgo.mesh.volume.VolumeMesh(
        pgo.mesh.CubicMeshData(vertices, elements),
        pgo.mesh.volume.ENuMaterial(),
    )
    rest_surface = rest_volume.extract_surface_mesh()
    seen = {}
    for layer, case in zip(layers, cases):
        if case.protocol not in seen:
            seen[case.protocol] = (layer, case)
    for protocol, color in (
            ("simple_shear", "lightsteelblue"),
            ("free_uniaxial", "lightcoral"),
            ("confined_uniaxial", "lightgreen")):
        layer, case = seen[protocol]
        layer(probe_params, plastic_empty)
        deformed = deformed_surface(
            operator, layer._last_displacement)
        pgo.visualize.plot_surface(
            [rest_surface, deformed],
            titles=["rest",
                    f"{case.protocol} ({case.coordinate:+.3f})"],
            colors=["lightgray", color],
            window_size=(800, 420),
            backend="jupyter",
        )
else:
    print("pyvista is not installed; skipping 3D visualization")


## 4. Generating the target data

The target Neo-Hookean material has zero optimizable channels, so its layer
input is `(num_elements, 0)`. We solve every load case with the target
operator and store the resulting reactions as `targets` -- our training
labels.

In [ ]:
target_layers = [
    StaticEquilibriumLayer(
        energy_operator=target_operator,
        fixed_dofs=case.fixed_dofs,
        fixed_values=case.fixed_values,
        reaction_selectors=case.reaction_selector,
        inner_optimizer=OPTIMIZER,
        sparse_backend=ps.EigenLDLT(),
        initial_displacement=case.initial_displacement,
        residual_tol=STATIC_TOL,
    )
    for case in cases
]

target_empty = torch.empty((num_elements, 0), dtype=torch.float64)
targets = torch.stack([
    layer(target_empty, target_empty) for layer in target_layers])

# Backfill the per-case labels so the plotting section can read them directly.
for case, value in zip(cases, targets):
    case.target = float(value.detach())

print(f"{len(cases)} training cases; first 6 normalized reactions:")
print(np.round((targets[:6] / FORCE_SCALE).numpy(), 5))

## 5. Residual and Jacobian via autograd

The fitted parameters are $p$ (f'' spline values + lambda), optimized in
**log space**: $\theta = \log(p / E_0)$ keeps parameters positive.

`reactions(theta)` stacks the differentiable reactions of all load cases. The
reaction Jacobian is obtained with `torch.autograd.functional.jacobian`, which
calls the layer's backward -- the implicit-function-theorem adjoint

$$K_{ff}^T \lambda = (K^T c)_f, \qquad
  \frac{dy}{d\theta} = \left(c^T E_{ue} - \lambda^T E_{ue,f}\right) p,$$

with $K = \frac{\partial^2 E}{\partial u^2}$ and $c$ the reaction selector.
No hand-written material-VJP bookkeeping is needed in user code; the layer
reuses the C++ `material_vjp` under the hood.

In [ ]:
def reactions(theta):
    """Stack the differentiable reaction of every load case."""
    params = FORCE_SCALE * torch.exp(theta)
    values = params.expand(num_elements, -1)
    return torch.stack([layer(values, plastic_empty) for layer in layers])


def residual_and_jacobian(theta):
    """Normalized residuals and their Jacobian (plus smoothness rows)."""
    residual = (reactions(theta) - targets) / FORCE_SCALE
    jacobian = torch.autograd.functional.jacobian(
        lambda t: (reactions(t) - targets) / FORCE_SCALE, theta)

    if SMOOTHNESS > 0.0:
        # Second-difference regularization on f'' (lambda excluded), weighted
        # by sqrt(w) and appended as extra least-squares rows.
        stretch_count = len(theta) - 1
        operator = torch.zeros(
            (stretch_count - 2, stretch_count), dtype=torch.float64)
        rows = torch.arange(stretch_count - 2)
        operator[rows, rows] = 1.0
        operator[rows, rows + 1] = -2.0
        operator[rows, rows + 2] = 1.0
        scale = np.sqrt(SMOOTHNESS)
        reg_jacobian = torch.zeros(
            (operator.shape[0], len(theta)), dtype=torch.float64)
        reg_jacobian[:, :-1] = scale * operator
        residual = torch.cat([residual, scale * operator @ theta[:-1]])
        jacobian = torch.cat([jacobian, reg_jacobian])
    return residual, jacobian


theta0 = torch.tensor(
    np.log(initial_params / FORCE_SCALE), dtype=torch.float64)

residual0, jacobian0 = residual_and_jacobian(
    theta0.clone().requires_grad_(True))
print(f"initial RMSE = "
      f"{torch.sqrt(torch.mean(residual0[:len(cases)] ** 2)):.6f}")
print(f"Jacobian shape = {tuple(jacobian0.shape)}")

## 6. Gauss-Newton fit

The outer loop solves the normal equations
$(J^T J + \text{damping}\,\mathrm{diag})\,\Delta\theta = -J^T r$. If the
full step does not decrease the objective, damping is increased
(Levenberg-Marquardt style) and the step is recomputed -- no line search is
needed. Every iteration re-solves all load cases through the layers; that is
still the main computational cost. The only difference from the NumPy
tutorial is that $J$ now comes from autograd instead of the explicit
`reaction_jacobian` function.

In [ ]:
def fit_parameters(theta0, max_iterations=40):
    theta = theta0.detach().clone().requires_grad_(True)
    damping = 1.0e-5
    history = []
    for iteration in range(max_iterations):
        residual, jacobian = residual_and_jacobian(theta)
        objective = 0.5 * float((residual @ residual).detach())
        gradient = jacobian.T @ residual
        train_rmse = float(
            torch.sqrt(torch.mean(residual[:len(cases)] ** 2)).detach())
        history.append({
            "iteration": iteration,
            "objective": objective,
            "train_rmse": train_rmse,
        })
        print(f"iter {iteration:2d}  objective={objective:.3e}  "
              f"train_rmse={train_rmse:.3e}")
        if torch.norm(gradient.detach(), p=float("inf")) < 1.0e-9:
            print("converged: gradient below tolerance")
            break
        normal = jacobian.T @ jacobian
        diagonal = torch.maximum(
            torch.diag(normal), torch.full((len(theta),), 1.0e-12))
        for _ in range(12):
            step = torch.linalg.solve(
                normal + damping * torch.diag(diagonal), -gradient)
            if torch.max(torch.abs(step)) > 0.75:
                step *= 0.75 / torch.max(torch.abs(step))
            candidate = theta + step
            candidate_residual, _ = residual_and_jacobian(candidate)
            candidate_objective = 0.5 * float(
                (candidate_residual @ candidate_residual).detach())
            if candidate_objective < objective:
                theta = candidate
                damping = max(damping / 3.0, 1.0e-12)
                break
            damping *= 10.0
        else:
            print("stopping: no decreasing step found")
            break
    return theta, history


theta, fit_history = fit_parameters(theta0)

## 7. Results and visualization

After the fit we check two things: how much the training RMSE dropped, and
how close the fitted f''/lambda values are to the closed-form Neo-Hookean
reference. Then we plot the reaction curves for the initial guess, the fitted
material, and the target, together with the optimization loss history.

In [ ]:
fitted = FORCE_SCALE * torch.exp(theta.detach()).numpy()
mu, lam = (TARGET_E / (2 * (1 + TARGET_NU)),
           TARGET_E * TARGET_NU / ((1 + TARGET_NU) * (1 - 2 * TARGET_NU)))
reference = np.concatenate([mu * (1 + 1 / knots ** 2), [lam]])

final_residual, _ = residual_and_jacobian(theta)
final_rmse = torch.sqrt(
    torch.mean(final_residual[:len(cases)] ** 2)).detach()
initial_rmse = torch.sqrt(
    torch.mean(residual0[:len(cases)] ** 2)).detach()
print(f"train RMSE: initial={float(initial_rmse):.6f}  "
      f"fitted={float(final_rmse):.6e}")
print(f"{'parameter':10s} {'fitted':>12s} {'reference':>12s}")
for i, name in enumerate([f"f_dd_{i}" for i in range(KNOT_COUNT)] + ["lambda"]):
    print(f"{name:10s} {fitted[i]:12.1f} {reference[i]:12.1f}")

In [ ]:
predicted = reactions(theta).detach().numpy()
initial_predicted = reactions(
    torch.tensor(np.log(initial_params / FORCE_SCALE), dtype=torch.float64)
).detach().numpy()

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

if HAS_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for ax, protocol in zip(
            axes, ("free_uniaxial", "confined_uniaxial", "simple_shear")):
        sel = np.array([i for i, c in enumerate(cases) if c.protocol == protocol])
        order = np.argsort([cases[i].coordinate for i in sel])
        sel = sel[order]
        x_plot = [cases[i].coordinate for i in sel]
        ax.plot(x_plot, [cases[i].target / FORCE_SCALE for i in sel],
                "o--", label="target (Neo-Hookean)")
        ax.plot(x_plot, [initial_predicted[i] / FORCE_SCALE for i in sel],
                "^:", color="gray", label="initial (Systematic Poking)")
        ax.plot(x_plot, [predicted[i] / FORCE_SCALE for i in sel],
                "s-", label="fitted (Systematic Poking)")
        ax.set_title(protocol)
        ax.set_xlabel("stretch or shear")
        ax.set_ylabel("reaction / E")
        ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()
else:
    print("matplotlib is not installed; skipping the plot")

In [ ]:
try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(6.5, 3.5))
    ax.semilogy(
        [h["iteration"] for h in fit_history],
        [h["objective"] for h in fit_history],
        "o-",
    )
    ax.set_xlabel("iteration")
    ax.set_ylabel("objective = 0.5 * ||residual||^2")
    ax.set_title("Optimization loss (Gauss-Newton, autograd Jacobian)")
    ax.grid(True, which="both", alpha=0.3)
    fig.tight_layout()
    plt.show()
else:
    print("matplotlib is not installed; skipping the loss plot")